# Phát hiện bot — toàn bộ pipeline trên Google Colab

Gộp nội dung notebook **01 → 02 → 03 → 04**. Các hàm dùng chung nằm trực tiếp
trong cell bên dưới; notebook chạy độc lập, không tạo thư mục `src/`.

1. Tải notebook này lên Colab, chọn runtime **Python / CPU** (không cần GPU).
2. Chọn **Run all**. Khi được yêu cầu, tải lên `twitter_human_bots_dataset.csv`.
3. Chờ chạy lần lượt: EDA → Isolation Forest / One-Class SVM → Gradient Boosting → đánh giá.
4. Cell cuối tạo và tải ZIP chứa báo cáo, biểu đồ, mô hình, đặc trưng và split.

Không cần tải 4 notebook gốc hoặc thư mục `src/`. Mã nguồn được nhúng là bản tại
thời điểm tạo notebook; sau khi sửa nguồn, chạy `python scripts/build_colab_notebook.py` để cập nhật.
Kết quả được ghi trong thư mục riêng `bot_detection_colab/` của runtime.
Runtime Colab là tạm thời: tải ZIP trước khi ngắt phiên. Muốn dùng lại phân hoạch cũ,
đặt `split.csv` vào `bot_detection_colab/dataset/processed/` trước phần 1;
nếu chưa có, pipeline tạo split bằng cùng seed 42 và quy tắc 60/20/20.
Các cấu hình/ngưỡng vẫn chỉ được chọn trên validation; test của dự án đã được xem
trong các thí nghiệm trước, nên không phải holdout hoàn toàn chưa từng quan sát.


## 0. Chuẩn bị môi trường và dữ liệu

In [ ]:
# Chạy cell này trước khi import các thư viện học máy.
%pip -q install "numpy>=2.3.3,<3" "pandas>=2.3.3,<4" "scikit-learn>=1.8,<2" "joblib>=1.5,<2" "matplotlib>=3.10,<4" "seaborn>=0.13.2,<0.14"


In [ ]:
from pathlib import Path
import shutil
import sys
from IPython.display import display

# Thư mục riêng giúp chạy local mà không ghi đè báo cáo gốc.
working_dir = Path.cwd().resolve()
project_root = working_dir / "bot_detection_colab"
project_root.mkdir(parents=True, exist_ok=True)
data_path = project_root / "dataset" / "twitter_human_bots_dataset.csv"
data_path.parent.mkdir(parents=True, exist_ok=True)
for candidate_root in (working_dir, working_dir.parent):
    candidate = candidate_root / "dataset" / data_path.name
    if not data_path.exists() and candidate.is_file() and candidate != data_path:
        shutil.copy2(candidate, data_path)
        old_split = candidate_root / "dataset" / "processed" / "split.csv"
        new_split = project_root / "dataset" / "processed" / "split.csv"
        if old_split.is_file() and not new_split.exists():
            new_split.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(old_split, new_split)

if not data_path.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"Đặt CSV nguồn tại {data_path} rồi chạy lại cell.") from None
    print("Chọn file twitter_human_bots_dataset.csv từ máy của bạn.")
    uploaded = files.upload()
    if data_path.name not in uploaded:
        raise ValueError(f"Cần tải lên đúng file {data_path.name}.")
    data_path.write_bytes(uploaded[data_path.name])
    del uploaded

print("Project root:", project_root)
print("CSV:", data_path)


### Cấu hình và các hàm dùng chung

Chạy các cell này trước bốn phần phân tích bên dưới.

In [ ]:
"""Đường dẫn và quy ước dùng chung cho mọi notebook."""

from pathlib import Path

PROJECT_ROOT = project_root
RAW_DATA_PATH = PROJECT_ROOT / "dataset" / "twitter_human_bots_dataset.csv"
FEATURES_PATH = PROJECT_ROOT / "dataset" / "processed" / "features.csv"
SPLIT_PATH = PROJECT_ROOT / "dataset" / "processed" / "split.csv"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

RANDOM_STATE = 42
LABEL_MAPPING = {"human": 0, "bot": 1}
SPLIT_NAMES = ("train", "validation", "test")  # 60 / 20 / 20

FEATURE_COLUMNS = [
    "follower_following_ratio",
    "posting_frequency",
    "account_age_days",
    "bio_length",
]


In [ ]:
"""Đọc dữ liệu, tạo 4 đặc trưng và chia tập train/validation/test."""

import pandas as pd
from sklearn.model_selection import train_test_split


NUMERIC_COLUMNS = ["followers_count", "friends_count", "statuses_count", "account_age_days"]


def load_profiles(path=RAW_DATA_PATH) -> pd.DataFrame:
    """Đọc CSV; dừng nếu ID/nhãn lỗi. Số không hợp lệ hoặc âm thành giá trị thiếu."""
    # keep_default_na=False: giữ nguyên bio là chữ "None"/"NA" thay vì biến thành thiếu.
    df = pd.read_csv(path, dtype={"id": "string", "description": "string"}, keep_default_na=False)
    missing = {"id", "account_type", "description", *NUMERIC_COLUMNS} - set(df.columns)
    if missing:
        raise ValueError(f"Thiếu cột: {sorted(missing)}")
    if df["id"].str.strip().eq("").any() or df["id"].duplicated().any():
        raise ValueError("ID rỗng hoặc bị trùng.")
    if not df["account_type"].isin(LABEL_MAPPING).all():
        raise ValueError(f"Nhãn phải là {list(LABEL_MAPPING)}.")
    for column in NUMERIC_COLUMNS:
        values = pd.to_numeric(df[column], errors="coerce")
        df[column] = values.mask(values < 0)
    df["label"] = df["account_type"].map(LABEL_MAPPING)
    return df.rename(columns={"id": "user_id"})


def build_features(profiles: pd.DataFrame) -> pd.DataFrame:
    """Tính 4 đặc trưng. Mẫu số bằng 0 thì dùng 1; mẫu số thiếu thì kết quả thiếu."""
    features = pd.DataFrame({"user_id": profiles["user_id"]})
    features["follower_following_ratio"] = profiles["followers_count"] / profiles["friends_count"].clip(lower=1)
    features["posting_frequency"] = profiles["statuses_count"] / profiles["account_age_days"].clip(lower=1)
    features["account_age_days"] = profiles["account_age_days"]
    features["bio_length"] = profiles["description"].str.len()  # bio rỗng có độ dài 0
    features["label"] = profiles["label"]
    return features


def split_features(features: pd.DataFrame, path=SPLIT_PATH):
    """Trả về (train, validation, test). Lần đầu chia 60/20/20 và lưu split.csv; các lần sau dùng lại."""
    if path.exists():
        split = pd.read_csv(path, dtype={"user_id": "string"}).set_index("user_id")["split"]
        if set(split.index) != set(features["user_id"]):
            raise ValueError(f"{path} không khớp với dữ liệu hiện tại.")
    else:
        rest, test = train_test_split(features, test_size=0.2, stratify=features["label"], random_state=RANDOM_STATE)
        train, val = train_test_split(rest, test_size=0.25, stratify=rest["label"], random_state=RANDOM_STATE)
        split = pd.concat([pd.Series(name, index=part["user_id"]) for name, part in
                           zip(SPLIT_NAMES, (train, val, test))]).rename("split")
        path.parent.mkdir(parents=True, exist_ok=True)
        split.rename_axis("user_id").reset_index().to_csv(path, index=False)
    assigned = features["user_id"].map(split)
    return tuple(features[assigned == name] for name in SPLIT_NAMES)


def run_pipeline():
    """Chạy bằng: python -m src.data  → lưu dataset/processed/features.csv."""
    features = build_features(load_profiles())
    FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)
    features.to_csv(FEATURES_PATH, index=False)
    return features




In [ ]:
"""Huấn luyện 3 mô hình, tìm cấu hình và ngưỡng trên validation, lưu/tải model.

Mọi hàm nhận cả bảng (train/validation/test) và tự lấy FEATURE_COLUMNS,
nên user_id và label không bao giờ lọt vào mô hình.
"""

from itertools import product

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import OneClassSVM


DETECTORS = ("isolation_forest", "one_class_svm")


def train_isolation_forest(train, n_estimators=300):
    """Không dùng nhãn. Mô hình cây nên không cần log/scale."""
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        IsolationForest(n_estimators=n_estimators, random_state=RANDOM_STATE, n_jobs=-1),
    )
    return model.fit(train[FEATURE_COLUMNS])


def train_one_class_svm(train, nu=0.1, gamma="scale"):
    """Không dùng nhãn. log1p + scale vì SVM đo khoảng cách và ratio lệch tới hàng triệu."""
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(np.log1p),
        StandardScaler(),
        OneClassSVM(kernel="rbf", nu=nu, gamma=gamma),
    )
    return model.fit(train[FEATURE_COLUMNS])


def train_gradient_boosting(train, n_estimators=100, max_depth=3):
    """Có dùng nhãn."""
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        GradientBoostingClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                   min_samples_leaf=5, random_state=RANDOM_STATE),
    )
    return model.fit(train[FEATURE_COLUMNS], train["label"])


def is_detector(model):
    """Isolation Forest / One-Class SVM không có predict_proba."""
    return not hasattr(model, "predict_proba")


def bot_scores(model, frame):
    """Điểm liên tục, càng lớn càng giống bot (dùng cho AP, ROC-AUC, chọn ngưỡng)."""
    X = frame[FEATURE_COLUMNS]
    if is_detector(model):
        return -model.decision_function(X)   # detector: nhỏ = lạ → đảo dấu
    return model.predict_proba(X)[:, 1]      # classes_ luôn là [0, 1] → cột 1 là bot


def default_predictions(model, frame):
    """Quyết định mặc định: detector dùng predict() == -1 (bất thường), classifier dùng 0,5."""
    pred = model.predict(frame[FEATURE_COLUMNS])
    return (pred == -1).astype(int) if is_detector(model) else pred.astype(int)


def select_threshold(y_val, scores):
    """Ngưỡng cho F1 bot cao nhất. CHỈ gọi trên validation."""
    precision, recall, thresholds = precision_recall_curve(y_val, scores)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    return float(thresholds[np.argmax(f1[:-1])])


def search(train_fn, grid, train, validation):
    """Thử mọi tổ hợp trong grid, chọn theo AP validation (hòa thì giữ cấu hình đầu).

    Trả về (model tốt nhất, cấu hình tốt nhất, bảng kết quả).
    """
    rows, best = [], None
    for values in product(*grid.values()):
        config = dict(zip(grid, values))
        model = train_fn(train, **config)
        ap = average_precision_score(validation["label"], bot_scores(model, validation))
        rows.append({**config, "ap_validation": ap})
        if best is None or ap > best[2]:
            best = (model, config, ap)
    table = pd.DataFrame(rows).sort_values("ap_validation", ascending=False, ignore_index=True)
    return best[0], best[1], table


def save_model(name, model, config, threshold):
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump({"model": model, "config": config, "threshold": threshold,
                 "feature_columns": FEATURE_COLUMNS}, MODELS_DIR / f"{name}.joblib")


def load_model(name):
    bundle = joblib.load(MODELS_DIR / f"{name}.joblib")
    if bundle["feature_columns"] != FEATURE_COLUMNS:
        raise ValueError(f"{name}: model được train với bộ đặc trưng khác; hãy train lại.")
    return bundle


In [ ]:
"""Tính chỉ số, so sánh các mô hình và feature importance. Quy ước: human=0, bot=1."""

import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)



def evaluate_predictions(y_true, y_pred, scores) -> dict:
    """Precision/Recall/F1 lớp bot, Macro-F1, AP, ROC-AUC, Accuracy và confusion matrix."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "precision_bot": precision_score(y_true, y_pred, zero_division=0),
        "recall_bot": recall_score(y_true, y_pred, zero_division=0),
        "f1_bot": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", labels=[0, 1], zero_division=0),
        "ap": average_precision_score(y_true, scores),
        "roc_auc": roc_auc_score(y_true, scores),
        "accuracy": accuracy_score(y_true, y_pred),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def compare_models(bundles: dict, frame: pd.DataFrame, split_name: str) -> pd.DataFrame:
    """Mỗi mô hình 2 dòng: ngưỡng mặc định và ngưỡng đã chọn trên validation. Không fit lại."""
    y, rows = frame["label"], []
    for name, bundle in bundles.items():
        model, scores = bundle["model"], bot_scores(bundle["model"], frame)
        for setting, pred in (("default", default_predictions(model, frame)),
                              ("validation_f1", (scores >= bundle["threshold"]).astype(int))):
            rows.append({"split": split_name, "model": name, "setting": setting,
                         "threshold": None if setting == "default" else bundle["threshold"],
                         **evaluate_predictions(y, pred, scores)})
    return pd.DataFrame(rows)


def feature_importance(model, validation: pd.DataFrame) -> pd.DataFrame:
    """Impurity importance (học từ train) + permutation importance (AP giảm trên validation)."""
    perm = permutation_importance(model, validation[FEATURE_COLUMNS], validation["label"],
                                  scoring="average_precision", n_repeats=10, random_state=RANDOM_STATE)
    return pd.DataFrame({
        "feature": FEATURE_COLUMNS,
        "impurity_importance": model[-1].feature_importances_,
        "permutation_ap_drop": perm.importances_mean,
        "permutation_ap_drop_std": perm.importances_std,
    }).sort_values("permutation_ap_drop", ascending=False, ignore_index=True)


# Phần 1 — Kiểm tra dữ liệu và trích xuất đặc trưng

Khảo sát dữ liệu hồ sơ, kiểm tra giá trị thiếu và ID trùng, tạo bốn đặc trưng dùng cho mô hình. Sau khi chia train/validation/test, phân tích phân phối và tương quan trên tập train.

Chạy các phần trong notebook này theo thứ tự từ trên xuống.

In [ ]:
import sys
import pandas as pd

print("Python của notebook:", sys.executable)
print("Phiên bản pandas:", pd.__version__)

data_path = project_root / "dataset" / "twitter_human_bots_dataset.csv"
print("CSV tồn tại:", data_path.is_file())

df = pd.read_csv(data_path, dtype={"id": "string"})

print("Kích thước:", df.shape)
print("Tên cột:", df.columns.tolist())
print("Kiểu dữ liệu của id:", df["id"].dtype)
print("Phân bố nhãn:")
print(df["account_type"].value_counts(dropna=False))

df.head(3)

In [ ]:
required_columns = [
    "id", "account_type",
    "followers_count", "friends_count", "statuses_count",
    "created_at", "description",
    "account_age_days", "average_tweets_per_day",
]

print("Số giá trị thiếu theo cột:")
print(df[required_columns].isna().sum())

bio = df["description"].astype("string")

missing_bio = bio.isna()
blank_bio = bio.notna() & bio.fillna("").str.strip().eq("")

print("Bio null:", missing_bio.sum())
print("Bio không null nhưng rỗng/chỉ có khoảng trắng:", blank_bio.sum())
print("Tổng bio null hoặc rỗng:", (missing_bio | blank_bio).sum())

In [ ]:
bio_literal = pd.read_csv(
    data_path,
    usecols=["description"],
    dtype={"description": "string"},
    keep_default_na=False,
)["description"]

print("Bio là chuỗi rỗng:", bio_literal.eq("").sum())
print(
    "Bio rỗng hoặc chỉ có khoảng trắng:",
    bio_literal.str.strip().eq("").sum(),
)

print("Nội dung gốc của các bio trước đó bị nhận diện là thiếu:")
print(bio_literal.loc[df["description"].isna()].map(repr).value_counts())

In [ ]:
df["description"] = bio_literal.mask(
    bio_literal.eq(""),
    pd.NA,
)

print("Bio thiếu sau khi xử lý:", df["description"].isna().sum())
print("Bio chứa đúng chuỗi 'None':", df["description"].eq("None").sum())
print("Số dòng:", len(df))

In [ ]:
duplicate_mask = df["id"].duplicated(keep=False)

print("Số dòng thuộc các ID trùng:", duplicate_mask.sum())
print("Số ID khác nhau bị trùng:", df.loc[duplicate_mask, "id"].nunique())

labels_per_id = df.groupby("id")["account_type"].nunique(dropna=False)
conflicting_ids = labels_per_id[labels_per_id > 1].index

print("Số ID có nhãn mâu thuẫn:", len(conflicting_ids))

if duplicate_mask.any():
    display(
        df.loc[duplicate_mask, ["id", "account_type"]]
        .sort_values("id")
        .head(20)
    )

## Tạo đặc trưng

`run_pipeline()` đọc lại CSV bằng `load_profiles()`, tạo đặc trưng bằng `build_features()` và lưu vào `dataset/processed/features.csv`.

Pipeline kiểm tra cột bắt buộc, ID và nhãn; chuyển số âm hoặc không đọc được thành giá trị thiếu. Bio được giữ dưới dạng chuỗi, bao gồm nội dung `None` và `NA`.

In [ ]:

features = run_pipeline()
print("Đã lưu:", FEATURES_PATH, features.shape)
print("Giá trị thiếu:", features[FEATURE_COLUMNS].isna().sum().to_dict())
display(features.head())

### Ý nghĩa bốn đặc trưng

| Đặc trưng | Cách tính |
| --- | --- |
| `follower_following_ratio` | Số follower / số following |
| `posting_frequency` | Tổng số bài đăng / tuổi tài khoản tính theo ngày |
| `account_age_days` | Tuổi tài khoản lấy từ dữ liệu nguồn |
| `bio_length` | Độ dài cột `description`, tính cả khoảng trắng và xuống dòng |

Mẫu số bằng 0 được thay bằng 1; giá trị thiếu vẫn được giữ là thiếu. Bio rỗng có độ dài 0. Tuổi tài khoản giữ theo dữ liệu nguồn, không tính lại theo ngày chạy. `user_id` dùng để định danh và `label` là nhãn (`human = 0`, `bot = 1`).

## Chia tập dữ liệu

`split_features()` chia theo tài khoản với tỷ lệ **60% train, 20% validation, 20% test**, giữ tỷ lệ nhãn bằng stratify và seed 42. Phân chia được lưu tại `dataset/processed/split.csv` để các notebook dùng cùng dữ liệu. Nếu file đã lưu không khớp tập ID hiện tại, hàm báo lỗi.

In [ ]:

train, validation, test = split_features(features)
print("Split:", SPLIT_PATH)
display(pd.DataFrame({
    name: {"rows": len(part), "share": len(part) / len(features), "bot_share": part["label"].mean()}
    for name, part in zip(SPLIT_NAMES, (train, validation, test))
}).T)

## Phân tích trên tập train

Bảng tóm tắt gồm trung vị theo nhãn, giá trị lớn nhất, độ lệch trước/sau `log1p`, tỷ lệ giá trị 0 và ROC-AUC của từng đặc trưng. ROC-AUC dưới 0,5 cho biết bot có xu hướng nhận giá trị thấp hơn human; mức cách xa 0,5 phản ánh khả năng phân biệt của riêng đặc trưng đó.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_auc_score


eda_X = train[list(FEATURE_COLUMNS)].astype("float64")
eda_y = train["label"].to_numpy()

display(pd.DataFrame([{
    "feature": column,
    "median_human": eda_X.loc[eda_y == 0, column].median(),
    "median_bot": eda_X.loc[eda_y == 1, column].median(),
    "max": eda_X[column].max(),
    "skew": eda_X[column].skew(),
    "skew_log1p": np.log1p(eda_X[column]).skew(),
    "zero_share": eda_X[column].eq(0).mean(),
    "single_feature_roc_auc": roc_auc_score(eda_y, eda_X[column]),
} for column in FEATURE_COLUMNS]).round(3))

In [ ]:
SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": MUTED, "axes.edgecolor": GRID, "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": GRID,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11,
})
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

panels = {
    "follower_following_ratio": ("Tỷ lệ follower/following", "log1p(tỷ lệ)", True),
    "posting_frequency": ("Tần suất đăng bài", "log1p(bài/ngày)", True),
    "account_age_days": ("Tuổi tài khoản", "ngày", False),
    "bio_length": ("Độ dài bio", "số ký tự", False),
}
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax, (column, (title, xlabel, use_log)) in zip(axes.ravel(), panels.items()):
    values = np.log1p(eda_X[column]) if use_log else eda_X[column]
    bins = np.linspace(values.min(), values.quantile(0.999), 50)
    for label, name, color in ((0, "human", "#2a78d6"), (1, "bot", "#eb6834")):
        ax.hist(values[eda_y == label], bins=bins, density=True, histtype="step",
                linewidth=2, color=color, label=name)
    ax.set(title=title, xlabel=xlabel, ylabel="mật độ")
    ax.grid(axis="y", linewidth=0.8)
    ax.set_axisbelow(True)
axes[0, 0].legend(frameon=False)
fig.suptitle("Phân phối đặc trưng theo nhãn (tập train)", fontsize=12)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_feature_distributions_train.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
correlation = eda_X.corr(method="spearman")
cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
    "blue_red", ["#2a78d6", "#f0efec", "#e34948"],
)
fig, ax = plt.subplots(figsize=(6.2, 4.8))
ax.imshow(correlation, cmap=cmap, vmin=-1, vmax=1)
for (i, j), value in np.ndenumerate(correlation.to_numpy()):
    ax.text(j, i, f"{value:.2f}", ha="center", va="center", color=INK, fontsize=10)
ax.set_xticks(range(len(correlation)), correlation.columns, rotation=25, ha="right")
ax.set_yticks(range(len(correlation)), correlation.index)
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_title("Tương quan Spearman giữa các đặc trưng (tập train)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "eda_spearman_correlation_train.png", dpi=150, bbox_inches="tight")
plt.show()

### Nhận xét

- `follower_following_ratio` và `posting_frequency` lệch phải mạnh; `log1p` giúp giảm độ lệch của hai đặc trưng này.
- `log1p` làm tăng độ lệch tuyệt đối của `account_age_days` và `bio_length`. Pipeline One-Class SVM hiện dùng phép biến đổi này cho cả bốn đặc trưng.
- Bot có trung vị thấp hơn human ở cả bốn đặc trưng. ROC-AUC từng đặc trưng dưới 0,5 phù hợp với xu hướng này.
- Từng đặc trưng riêng lẻ chỉ phân biệt hai lớp ở mức vừa phải; cần đánh giá khi kết hợp trong mô hình.
- Tương quan Spearman giữa các cặp đặc trưng thấp, chưa cho thấy sự dư thừa rõ ràng.
- Các đỉnh phân phối khác nhau giữa hai nhãn, đặc biệt ở độ dài bio và tuổi tài khoản.

# Phần 2 — Phát hiện bất thường

Huấn luyện **Isolation Forest** và **One-Class SVM** trên tập train bằng bốn đặc trưng hồ sơ. Dùng validation để chọn cấu hình One-Class SVM và ngưỡng phân loại; lưu mô hình để đánh giá trong notebook 04.

Quy ước: `human = 0`, `bot = 1`. Điểm bất thường là `-decision_function`; điểm càng lớn càng bất thường.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


train, validation, test = split_features(build_features(load_profiles()))
y_val = validation["label"].to_numpy()
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
display(pd.DataFrame({
    name: {"rows": len(part), "bot_share": part["label"].mean()}
    for name, part in zip(SPLIT_NAMES, (train, validation, test))
}).T)

## Mốc so sánh trên validation

Hai mốc tối thiểu mà một detector cần vượt qua:

- **Xếp hạng ngẫu nhiên:** ROC-AUC = 0,5 và AP = tỷ lệ bot.
- **Gắn nhãn tất cả là bot:** recall = 1, precision = tỷ lệ bot; F1 lớp bot = 2p / (1 + p).

In [ ]:
def validation_metrics(y_pred, scores):
    """Shared evaluate_predictions() on validation, plus the share flagged as bot."""
    return {**evaluate_predictions(y_val, y_pred, scores), "flagged_share": float(np.mean(y_pred))}


bot_share = y_val.mean()
baselines = pd.DataFrame([
    {"model": "random ranking", "threshold": "-", "ap": bot_share, "roc_auc": 0.5},
    {"model": "all bot", "threshold": "-", **validation_metrics(np.ones_like(y_val), np.zeros(len(y_val)))},
])
baselines.loc[1, ["ap", "roc_auc"]] = [bot_share, 0.5]
display(baselines.round(3))

## Chọn cấu hình One-Class SVM trên validation

Thứ hạng điểm của One-Class SVM phụ thuộc vào `nu` và `gamma`, nên thử một lưới nhỏ và chọn theo **AP trên validation** (không phụ thuộc ngưỡng). Isolation Forest giữ cấu hình mặc định của `train_isolation_forest` (300 cây, `contamination="auto"`); `contamination` chỉ dời ngưỡng mặc định, không đổi thứ hạng điểm.

In [ ]:
svm, svm_config, svm_search = search(
    train_one_class_svm,
    {"nu": [0.05, 0.1, 0.2], "gamma": ["scale", 0.05, 1.0]},
    train, validation,
)
display(svm_search.round(4))
print("Cấu hình One-Class SVM được chọn:", svm_config)

## Fit Isolation Forest trên train

Mọi hàm huấn luyện nhận cả bảng `train` và chỉ lấy bốn cột `FEATURE_COLUMNS`. Pipeline mang theo imputer (và `log1p` + scaler cho One-Class SVM), nên tiền xử lý chỉ học từ train. Giữ lại SVM tốt nhất ở bước tìm kiếm, không fit lại.

In [ ]:
iso = train_isolation_forest(train, n_estimators=300)
models = {"isolation_forest": iso, "one_class_svm": svm}
configs = {"isolation_forest": {"n_estimators": 300}, "one_class_svm": svm_config}
val_scores = {name: bot_scores(model, validation) for name, model in models.items()}

## Ngưỡng phân loại

- **Mặc định:** dùng `predict()` của detector, ánh xạ `-1` thành bot.
- **Validation F1:** dùng `select_threshold()` để chọn ngưỡng tối đa F1 lớp bot trên validation.

Các detector được fit không dùng nhãn; bước chọn cấu hình SVM và ngưỡng validation F1 có sử dụng nhãn validation.

In [ ]:
for name, model in models.items():
    threshold = select_threshold(y_val, val_scores[name])
    save_model(name, model, configs[name], threshold)

results = compare_models({name: load_model(name) for name in DETECTORS}, validation, "validation")
results.to_csv(REPORTS_DIR / "anomaly_validation_metrics.csv", index=False)
display(results.round(3))

## Phân phối điểm bất thường theo nhãn

Nếu bot là điểm bất thường, phân phối điểm của bot phải lệch sang phải so với người thật.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, (name, scores) in zip(axes, val_scores.items()):
    for label, title in ((0, "human"), (1, "bot")):
        ax.hist(scores[y_val == label], bins=60, alpha=0.6, density=True, label=title)
    ax.axvline(0, color="black", linestyle="--", linewidth=1, label="ngưỡng mặc định")
    ax.set(title=name, xlabel="điểm bất thường (-decision_function)", ylabel="mật độ")
    ax.legend()
fig.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure_path = FIGURES_DIR / "anomaly_score_distribution_validation.png"
fig.savefig(figure_path, dpi=150)
print("Đã lưu:", figure_path.relative_to(project_root).as_posix())
plt.show()

## Tài khoản nào bị gắn cờ?

So sánh trung vị đặc trưng giữa tài khoản bị gắn cờ (ngưỡng mặc định) và không bị gắn cờ trên validation, cùng tỷ lệ bot thực tế trong mỗi nhóm.

In [ ]:
profile_rows = []
for name, model in models.items():
    flagged = default_predictions(model, validation)
    for value, group in ((1, "flagged"), (0, "not flagged")):
        part = validation[flagged == value]
        profile_rows.append({
            "model": name, "group": group, "rows": len(part), "bot_share": part["label"].mean(),
            **part[list(FEATURE_COLUMNS)].median().to_dict(),
        })
display(pd.DataFrame(profile_rows).round(2))

bot_medians = validation.groupby("label")[list(FEATURE_COLUMNS)].median().rename(index={0: "human", 1: "bot"})
display(bot_medians.round(2))

## Lưu kết quả

`save_model()` lưu pipeline, danh sách đặc trưng, cấu hình và ngưỡng vào `models/*.joblib`. Bảng đánh giá được lưu tại `reports/anomaly_validation_metrics.csv`; biểu đồ nằm trong `reports/figures/`.

## Nhận xét trên validation

Đối chiếu các bảng kết quả với hai mốc ngẫu nhiên và gắn nhãn tất cả là bot. Cấu hình và ngưỡng được chọn trên cùng tập validation, nên kết quả cần được đối chiếu với notebook 04.

- **Isolation Forest không phát hiện được bot:** ROC-AUC 0,378 và AP 0,288, đều thấp hơn mốc xếp hạng ngẫu nhiên (0,5 và 0,332). Các tài khoản bị gắn cờ có tỷ lệ follower/following trung vị 54, đăng 19,9 bài/ngày và bio dài 98 ký tự, giống tài khoản nổi tiếng/truyền thông hơn là bot; tỷ lệ bot trong nhóm này (0,32) không cao hơn tỷ lệ chung (0,33).
- **One-Class SVM chỉ nhỉnh hơn ngẫu nhiên một chút:** cấu hình tốt nhất (`nu=0.05`, `gamma=0.05`) đạt ROC-AUC 0,565 và AP 0,372.
- **Ngưỡng mặc định gắn cờ quá ít:** F1 lớp bot 0,152 (Isolation Forest) và 0,087 (One-Class SVM).
- **Ngưỡng chọn trên validation không thực sự cứu được:** Isolation Forest suy biến thành "tất cả là bot" (F1 0,498); One-Class SVM gắn cờ 82,7% tài khoản và đạt F1 0,501, gần như bằng mốc "tất cả là bot".

**Giải thích:** trong dữ liệu này bot chiếm 33% và có hồ sơ khá "điển hình" (trung vị tỷ lệ follower/following 1,01, bio 35 ký tự), nên chúng nằm trong vùng dày đặc của không gian đặc trưng chứ không phải điểm ngoại lai. Điểm ngoại lai thực sự chủ yếu là tài khoản người thật có lượng follower rất lớn. Vì vậy, với bốn đặc trưng hồ sơ này, giả định "bot = bất thường" không đứng vững; mô hình có giám sát (notebook 03) là mốc cần so sánh.

# Phần 3 — Phân loại bằng Gradient Boosting

Huấn luyện Gradient Boosting với bốn đặc trưng hồ sơ, quy ước `human = 0`, `bot = 1`. Chọn cấu hình theo **AP trên validation**, sau đó chọn ngưỡng tối đa **F1 lớp bot**.

Mô hình được fit trên train và lưu để đánh giá trong notebook 04. Chạy lại notebook sẽ ghi đè các tệp mô hình và báo cáo tương ứng.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay


print("Project root:", project_root)

## Cấu hình và tiền xử lý

Thử **100/200 cây × độ sâu 2/3/4**, với learning rate 0,1, tối thiểu 5 mẫu/lá, subsample=1 và seed 42. Nếu AP bằng nhau, giữ cấu hình xuất hiện trước trong lưới. Nếu nhiều ngưỡng đạt cùng F1, chọn ngưỡng nhỏ nhất.

Pipeline điền giá trị thiếu bằng trung vị học từ train. Gradient Boosting không cần chuẩn hóa thang đo hoặc `log1p`. Giữ mô hình đã chọn để ngưỡng validation tương ứng với cùng cách tính điểm.

In [ ]:
train, validation, _ = split_features(build_features(load_profiles()))
gb, gb_config, gb_search = search(
    train_gradient_boosting,
    {"n_estimators": [100, 200], "max_depth": [2, 3, 4]},
    train, validation,
)
display(gb_search.round(4))
save_model("gradient_boosting", gb, gb_config,
           select_threshold(validation["label"], bot_scores(gb, validation)))

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
gb_search.to_csv(REPORTS_DIR / "supervised_validation_search.csv", index=False)
results = compare_models({"gradient_boosting": load_model("gradient_boosting")}, validation, "validation")
results.to_csv(REPORTS_DIR / "supervised_validation_metrics.csv", index=False)

## Kết quả trên validation

So sánh ngưỡng mặc định 0,5 với ngưỡng tối đa F1 trên validation. Vì validation cũng dùng để chọn cấu hình và ngưỡng, kết quả tại đây phục vụ phát triển mô hình; đánh giá test nằm ở notebook 04.

In [ ]:
display(results.round(6))
chosen = results[results["setting"].eq("validation_f1")].iloc[0]
display(Markdown(
    f"Cấu hình chọn: **{gb_config['n_estimators']} cây, max_depth={gb_config['max_depth']}**; "
    f"ngưỡng **{chosen['threshold']:.6f}**. "
    f"Validation: F1 bot **{chosen['f1_bot']:.4f}**, AP **{chosen['ap']:.4f}**, "
    f"ROC-AUC **{chosen['roc_auc']:.4f}**. "
))

## Precision–Recall và confusion matrix trên validation

Ma trận dùng ngưỡng đã lưu trong bundle, thứ tự lớp human/bot. Đường Precision–Recall dùng xác suất bot, không dùng nhãn dự đoán để tính AP.

In [ ]:
y_val = validation["label"].to_numpy()
scores = bot_scores(gb, validation)
labels = (scores >= chosen["threshold"]).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
PrecisionRecallDisplay.from_predictions(y_val, scores, name="gradient_boosting", ax=axes[0])
axes[0].axhline(y_val.mean(), color="gray", linestyle="--", label="Tỷ lệ bot")
axes[0].set_title("Precision–Recall (validation)")
axes[0].legend()
ConfusionMatrixDisplay.from_predictions(y_val, labels, labels=[0, 1],
                                       display_labels=["human", "bot"], ax=axes[1], colorbar=False)
axes[1].set_title("gradient_boosting (ngưỡng lưu)")
fig.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure_path = FIGURES_DIR / "supervised_validation_diagnostics.png"
fig.savefig(figure_path, dpi=150)
plt.show()
print("Đã lưu:", figure_path.relative_to(project_root))

## Tệp đầu ra

- `models/gradient_boosting.joblib`: pipeline, cấu hình, ngưỡng và danh sách đặc trưng.
- `reports/supervised_validation_search.csv`: kết quả tìm kiếm cấu hình.
- `reports/supervised_validation_metrics.csv`: chỉ số đánh giá trên validation.
- `reports/figures/supervised_validation_diagnostics.png`: đường Precision–Recall và ma trận nhầm lẫn.

Tiếp tục với notebook **04_evaluation.ipynb** để so sánh ba mô hình trên cùng tập test.

# Phần 4 — Đánh giá và giải thích mô hình

So sánh **Isolation Forest, One-Class SVM và Gradient Boosting** trên cùng tập test. Notebook tải các mô hình cùng cấu hình và ngưỡng đã lưu từ notebook 02–03.

Quy ước: `human = 0`, `bot = 1`; điểm càng lớn càng được mô hình xem là giống bot. Chạy notebook 02 và 03 trước để tạo các tệp trong `models/`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve


_, validation, test = split_features(build_features(load_profiles()))
y_test = test["label"].to_numpy()
bundles = {name: load_model(name) for name in ("isolation_forest", "one_class_svm", "gradient_boosting")}
display(pd.DataFrame({
    name: {"config": bundle["config"], "validation_f1_threshold": bundle["threshold"]}
    for name, bundle in bundles.items()
}).T)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

## Các thiết lập đánh giá

Mỗi mô hình được đánh giá với quyết định mặc định và ngưỡng F1 đã chọn trên validation. Các chỉ số gồm precision, recall, F1 lớp bot, Macro-F1, AP, ROC-AUC và accuracy. Giữ nguyên mô hình và ngưỡng khi chuyển từ validation sang test.

In [ ]:
validation_results = compare_models(bundles, validation, "validation")
validation_results.to_csv(REPORTS_DIR / "comparison_validation_metrics.csv", index=False)

# Chỉ đánh giá test sau khi cấu hình và ngưỡng đã cố định trên validation.
metrics = compare_models(bundles, test, "test")
metrics.to_csv(REPORTS_DIR / "metrics.csv", index=False)
print("Đã lưu các bảng so sánh vào reports/.")

bot_share = y_test.mean()
print(f"Mốc trên test: tỷ lệ bot {bot_share:.3f}; tất cả là bot cho F1 {2 * bot_share / (1 + bot_share):.3f}")
display(metrics.round(4))

## So sánh validation và test

Đối chiếu F1 lớp bot, Macro-F1, AP và ROC-AUC giữa hai tập để xem mức chênh lệch hiệu quả của từng mô hình.

In [ ]:
keys = ["model", "setting"]
comparison = validation_results[keys + ["f1_bot", "macro_f1", "ap", "roc_auc"]].merge(
    metrics[keys + ["f1_bot", "macro_f1", "ap", "roc_auc"]], on=keys, suffixes=("_val", "_test"),
)
display(comparison.round(3))

## Biểu đồ

Biểu đồ so sánh ba mô hình; đường tham chiếu màu xám biểu thị tỷ lệ bot. Số liệu đầy đủ nằm trong `reports/metrics.csv`.

In [ ]:
SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
MODEL_COLORS = {"gradient_boosting": "#2a78d6", "isolation_forest": "#eb6834", "one_class_svm": "#1baf7a"}
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": MUTED, "axes.edgecolor": GRID, "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": GRID,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11,
})
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, filename):
    fig.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches="tight")
    print("Đã lưu:", (FIGURES_DIR / filename).relative_to(project_root).as_posix())

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ap_by_model = metrics.groupby("model")["ap"].first()
label_points = {"gradient_boosting": (0.62, 8), "one_class_svm": (0.72, 7), "isolation_forest": (0.72, -14)}
for name, color in MODEL_COLORS.items():
    precision, recall, _ = precision_recall_curve(y_test, bot_scores(bundles[name]["model"], test))
    legend_label = f"{name} (AP {ap_by_model[name]:.3f})"
    ax.plot(recall, precision, color=color, linewidth=2, label=legend_label)
    target_recall, offset = label_points[name]
    point = np.argmin(np.abs(recall - target_recall))
    ax.annotate(name, (recall[point], precision[point]), xytext=(0, offset), textcoords="offset points",
                color=INK, ha="left" if name == "gradient_boosting" else "center", fontsize=9)
ax.axhline(bot_share, color=MUTED, linestyle="--", linewidth=1.5, label=f"Mốc ngẫu nhiên (AP {bot_share:.3f})")
ax.set(title="Đường Precision–Recall trên tập test", xlabel="Recall (lớp bot)", ylabel="Precision (lớp bot)",
       xlim=(0, 1), ylim=(0, 1.02))
ax.grid(axis="y", linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2)
save_figure(fig, "precision_recall_test.png")
plt.show()

In [ ]:
panels = [
    ("isolation_forest", "default"),
    ("one_class_svm", "default"), ("gradient_boosting", "validation_f1"),
]
BLUE_RAMP = ["#cde2fb", "#86b6ef", "#2a78d6", "#104281"]
cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list("blue", BLUE_RAMP)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharey=True)
for ax, (name, setting) in zip(axes, panels):
    row = metrics[(metrics["model"] == name) & (metrics["setting"] == setting)].iloc[0]
    matrix = np.array([[row["tn"], row["fp"]], [row["fn"], row["tp"]]])
    ax.imshow(matrix, cmap=cmap, vmin=0, vmax=len(y_test) * (1 - bot_share))
    for (i, j), count in np.ndenumerate(matrix):
        dark_cell = count > 0.5 * len(y_test) * (1 - bot_share)
        ax.text(j, i, f"{count:,}".replace(",", "."), ha="center", va="center",
                color="white" if dark_cell else INK, fontsize=11)
    ax.set_title(f"{name}\n{setting}", fontsize=10)
    ax.set_xticks([0, 1], ["human", "bot"])
    ax.set_yticks([0, 1], ["human", "bot"])
    ax.set_xlabel("Dự đoán")
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
axes[0].set_ylabel("Nhãn thật")
fig.suptitle("Ma trận nhầm lẫn trên tập test (cùng thang màu)", y=1.06, fontsize=11)
save_figure(fig, "confusion_matrices_test.png")
plt.show()

## Ảnh hưởng của đặc trưng trong Gradient Boosting

- `feature_importances_`: mức giảm impurity trên **train**; không phụ thuộc dữ liệu đánh giá nhưng có thể thiên về đặc trưng nhiều giá trị khác nhau.
- Permutation importance: mức giảm AP trên **validation** khi xáo trộn một đặc trưng (10 lần lặp). Test không được dùng cho phần này.

Importance cho biết mô hình dựa vào đặc trưng nào, không phải quan hệ nhân quả.

In [ ]:
importance = feature_importance(bundles["gradient_boosting"]["model"], validation)
importance.to_csv(REPORTS_DIR / "feature_importance.csv", index=False)
display(importance.round(4))

order = importance.iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
panels = (
    ("impurity_importance", None, "Giảm impurity (train)", "{:.3f}"),
    ("permutation_ap_drop", "permutation_ap_drop_std", "Mức giảm AP khi xáo trộn (validation)", "{:.3f}"),
)
for ax, (column, error, title, fmt) in zip(axes, panels):
    bars = ax.barh(order["feature"], order[column], height=0.5, color="#2a78d6",
                   xerr=order[error] if error else None, error_kw={"ecolor": MUTED, "linewidth": 1})
    offset = (order[error].to_numpy() if error else np.zeros(len(order)))
    for bar, extra, value in zip(bars, offset, order[column]):
        ax.text(value + extra + order[column].max() * 0.02, bar.get_y() + bar.get_height() / 2,
                fmt.format(value), va="center", fontsize=9)
    ax.set_title(title)
    ax.set_xlim(0, order[column].max() * 1.2)
    ax.grid(axis="x", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", length=0)
fig.tight_layout()
save_figure(fig, "feature_importance_gb.png")
plt.show()

## Kết luận và giới hạn

Bảng tổng hợp dưới đây sử dụng ngưỡng F1 đã chọn trên validation. Đối chiếu AP, F1 lớp bot và Macro-F1 để so sánh ba mô hình; tham khảo thêm ma trận nhầm lẫn và mốc gắn nhãn tất cả là bot.

Các giới hạn cần lưu ý khi diễn giải kết quả:

- Chỉ dùng bốn đặc trưng hồ sơ, chưa có nội dung bài đăng hoặc mạng lưới tương tác.
- Tần suất đăng là trung bình cả vòng đời; tuổi tài khoản lấy từ dữ liệu nguồn, không phản ánh trực tiếp hoạt động gần đây.
- Kết quả thuộc một bộ dữ liệu và một lần chia ngẫu nhiên, chưa có khoảng tin cậy hoặc đánh giá theo thời gian.
- Feature importance thể hiện mức mô hình sử dụng đặc trưng, không chứng minh quan hệ nhân quả.
- Dự đoán có thể nhầm; kết quả phân loại bot không đủ để kết luận một tài khoản giả mạo hay lừa đảo.

In [ ]:
from IPython.display import Markdown

selected = metrics[metrics["setting"].eq("validation_f1")]
display(selected[["model", "setting", "precision_bot", "recall_bot", "f1_bot", "macro_f1", "ap", "roc_auc", "accuracy"]].round(4))
gb = selected[selected["model"].eq("gradient_boosting")].iloc[0]
display(Markdown(
    f"Gradient Boosting trên test: **F1 bot {gb['f1_bot']:.4f}**, "
    f"**AP {gb['ap']:.4f}**, **ROC-AUC {gb['roc_auc']:.4f}**. "
    f"Đặc trưng có permutation importance lớn nhất: **{importance.iloc[0]['feature']}** "
    f"(AP giảm {importance.iloc[0]['permutation_ap_drop']:.4f} trên validation). "
    "Cấu hình và ngưỡng giữ nguyên từ notebook 03."
))

## 5. Tải kết quả

ZIP chứa `reports/`, `models/` và `dataset/processed/`.
CSV nguồn không nằm trong ZIP. Lưu lại `split.csv` nếu cần tái sử dụng phân hoạch.


In [ ]:
import zipfile

archive_base = project_root / "bot_detection_results"
with zipfile.ZipFile(archive_base.with_suffix(".zip"), "w",
                     compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in ("reports", "models", "dataset/processed"):
        for path in sorted((project_root / folder).rglob("*")):
            if path.is_file() and "__pycache__" not in path.parts:
                archive.write(path, path.relative_to(project_root))
archive_path = archive_base.with_suffix(".zip")
print("Đã tạo:", archive_path)
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink
    display(FileLink(str(archive_path)))
else:
    files.download(str(archive_path))
